# Benchmark monthly cross-sectional scaling

This notebook benchmarks `scale_chars_cross_sectionally_by_month_bigdata_v2` against `scale_chars_cross_sectionally_by_month`.

The synthetic panel contains 60 years of monthly observations and `n_firms_per_month` cross-sectional observations per month. A single random characteristic is duplicated: each implementation transforms one copy.

`date` has dtype `period[M]`, so every unique date value is exactly one monthly cross-section.

In [ ]:
import gc
import time
from statistics import median

import numpy as np
import pandas as pd
from sklearn.preprocessing import QuantileTransformer


In [ ]:
def scale_chars_cross_sectionally_by_month_bigdata_v2(
    df: pd.DataFrame,
    characteristic_cols: list[str],
    n_quantiles: int = 1000,
    random_state: int = 42,
) -> pd.DataFrame:
    """Fit a separate QuantileTransformer for every (month, characteristic)."""
    out = df.copy()

    for col in characteristic_cols:
        result = np.full(len(out), np.nan, dtype=np.float64)

        for _, group in out.groupby("date", sort=False):
            positions = group.index.to_numpy()
            values = group[col].to_numpy(dtype=np.float64, copy=False)

            valid = np.isfinite(values)
            n_valid = valid.sum()
            if n_valid == 0:
                continue
            if n_valid == 1:
                result[positions[valid]] = 0.0
                continue

            transformer = QuantileTransformer(
                n_quantiles=min(n_quantiles, n_valid),
                output_distribution="uniform",
                random_state=random_state,
                subsample=None,
            )
            result[positions[valid]] = transformer.fit_transform(
                values[valid].reshape(-1, 1)
            ).ravel()

        out[col] = 2.0 * result - 1.0

    return out

In [ ]:
def scale_chars_cross_sectionally_by_month(
    df: pd.DataFrame,
    characteristic_cols: list[str],
    n_quantiles: int = 1000,
    random_state: int = 42,
) -> pd.DataFrame:
    """Use groupby.transform with a fresh transformer per group."""
    out = df.copy()

    for col in characteristic_cols:
        def transform_month(s: pd.Series) -> np.ndarray:
            values = s.to_numpy(dtype=np.float64, copy=False)
            valid = np.isfinite(values)
            n_valid = valid.sum()
            result = np.full(len(s), np.nan, dtype=np.float64)

            if n_valid == 0:
                return result
            if n_valid == 1:
                result[valid] = 0.0
                return result

            transformer = QuantileTransformer(
                n_quantiles=min(n_quantiles, n_valid),
                output_distribution="uniform",
                random_state=random_state,
                subsample=None,
            )
            result[valid] = transformer.fit_transform(
                values[valid].reshape(-1, 1)
            ).ravel()
            return 2.0 * result - 1.0

        out[col] = out.groupby("date", sort=False)[col].transform(transform_month)

    return out


In [ ]:
# Increase this to make the benchmark heavier.
n_years = 60
n_firms_per_month = 5_000
seed = 42

rng = np.random.default_rng(seed)
months = pd.period_range("1960-01", periods=n_years * 12, freq="M")
date = np.repeat(months, n_firms_per_month)

df = pd.DataFrame({
    "date": pd.Series(date, dtype="period[M]"),
    "characteristic_groupby": rng.standard_normal(len(date)),
})
df["characteristic_loop"] = df["characteristic_groupby"].copy()

print(f"Rows: {len(df):,}")
print(f"Months: {df['date'].nunique():,}")
print(f"Firms per month: {n_firms_per_month:,}")
print(df.dtypes)
df.head()

In [ ]:
def benchmark(func, column: str, repeats: int = 3) -> tuple[pd.DataFrame, list[float]]:
    times = []
    output = None

    for _ in range(repeats):
        start = time.perf_counter()
        output = func(
            df,
            characteristic_cols=[column],
            n_quantiles=1000,
            random_state=42,
        )
        times.append(time.perf_counter() - start)
        gc.collect()

    return output, times

result_groupby, times_groupby = benchmark(
    scale_chars_cross_sectionally_by_month,
    column="characteristic_groupby",
)

result_loop, times_loop = benchmark(
    scale_chars_cross_sectionally_by_month_bigdata_v2,
    column="characteristic_loop",
)

summary = pd.DataFrame({
    "function": [
        "scale_chars_cross_sectionally_by_month",
        "scale_chars_cross_sectionally_by_month_bigdata_v2",
    ],
    "median_seconds": [median(times_groupby), median(times_loop)],
    "min_seconds": [min(times_groupby), min(times_loop)],
    "max_seconds": [max(times_groupby), max(times_loop)],
    "all_runs_seconds": [times_groupby, times_loop],
})
summary["relative_to_fastest"] = summary["median_seconds"] / summary["median_seconds"].min()
summary

In [ ]:
# Correctness checks: the two identical input columns should yield identical scaled values.
scaled_groupby = result_groupby["characteristic_groupby"].to_numpy()
scaled_loop = result_loop["characteristic_loop"].to_numpy()

assert np.allclose(scaled_groupby, scaled_loop, equal_nan=True)
assert np.nanmin(scaled_groupby) >= -1.0
assert np.nanmax(scaled_groupby) <= 1.0

monthly_bounds = (
    result_groupby.groupby("date", observed=True)["characteristic_groupby"]
    .agg(["min", "max"])
)

print("Outputs match and lie in [-1, 1].")
monthly_bounds.head()